# Filter records by categories and selecting diverse code submissions

In [1]:
!pip install datasketch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 2.7 MB/s eta 0:00:00


## Import libraries

In [2]:
from datasets import Dataset, load_dataset,concatenate_datasets
from sentence_transformers import SentenceTransformer, util
from collections import defaultdict
import torch
import pandas as pd
from datasketch import MinHashLSH, MinHash
import re
from tqdm.auto import tqdm
import numpy as np

2026-02-06 06:54:04.796397: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770360845.034922      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770360845.104801      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770360845.718412      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770360845.718467      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770360845.718470      55 computation_placer.cc:177] computation placer alr

## Load dataset

In [3]:
ds = load_dataset(
    "json",
    data_files="/kaggle/input/bytedance-new/code_contests_plus.jsonl",
    split="train"
)


Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/35 [00:00<?, ?it/s]

In [4]:
ds

Dataset({
    features: ['source', 'id', 'title', 'description', 'time_limit', 'memory_limit', 'correct_submissions', 'incorrect_submissions', 'category'],
    num_rows: 11690
})

## Count records in each category

In [5]:
cat_counts = pd.Series(ds["category"]).value_counts()
cat_counts = cat_counts.reset_index()
cat_counts.columns = ["category", "count"]
cat_counts.to_csv('category_count.csv')

## Filter data with chosen categories

In [6]:
TARGET = [
    "Graph Theory", "Sorting", "String", "Greedy", "Math", "Tree", "Counting",
    "Dynamic Programming", "Matrix", "Array", "Backtracking", "Permutation",
    "Divide and", "Hash Table", "Simulation"
]

buckets = defaultdict(list)

for row in ds:
    cat = row["category"]
    if cat in TARGET and len(buckets[cat]) < 75:
        buckets[cat].append(row)

subsets = [Dataset.from_list(rows) for rows in buckets.values()]

filtered = concatenate_datasets(subsets)

print(filtered)
print("Total samples:", len(filtered))


Dataset({
    features: ['source', 'id', 'title', 'description', 'time_limit', 'memory_limit', 'correct_submissions', 'incorrect_submissions', 'category'],
    num_rows: 1078
})
Total samples: 1078


## Using MinHash and Local Sensitive Hashing (LSH) to select diverse code submissions

In [7]:
ALLOWED_LANGS = {"cpp", "java", "py3"}

def simple_tokenize(code):
    return re.findall(r"[A-Za-z_]\w+|\S", code)

def minhash_signature(tokens, num_perm=128):
    m = MinHash(num_perm=num_perm)
    for t in tokens:
        m.update(t.encode("utf8"))
    return m

def pick_diverse_submissions_lsh(submissions, k=10, threshold=0.85):
    if not submissions:
        return []

    filtered = [s for s in submissions if s.get("language") in ALLOWED_LANGS]

    if not filtered:
        return []

    lsh = MinHashLSH(threshold=threshold, num_perm=128)

    picked = []
    for i, sub in enumerate(filtered):
        tokens = simple_tokenize(sub["code"])
        m = minhash_signature(tokens)

        duplicated = False
        for key in lsh.query(m):
            duplicated = True
            break

        if not duplicated:
            key = f"sub_{i}"
            lsh.insert(key, m)
            picked.append(sub)

        if len(picked) >= k:
            break

    return picked

In [8]:
results = []

for i, row in tqdm(enumerate(filtered), total=len(filtered), desc="Processing rows"):
    correct = row["correct_submissions"]
    incorrect = row["incorrect_submissions"]

    row["correct_submissions"] = pick_diverse_submissions_lsh(correct, k=10)
    row["incorrect_submissions"] = pick_diverse_submissions_lsh(incorrect, k=10)

    results.append(row)

final_dataset = Dataset.from_list(results)


Processing rows:   0%|          | 0/1078 [00:00<?, ?it/s]

## Check average code records in each submission

In [9]:
correct_counts = []
incorrect_counts = []

for row in final_dataset:
    correct_counts.append(len(row["correct_submissions"]))
    incorrect_counts.append(len(row["incorrect_submissions"]))

avg_correct = np.mean(correct_counts)
avg_incorrect = np.mean(incorrect_counts)

avg_correct, avg_incorrect


(np.float64(9.233766233766234), np.float64(8.943413729128014))

## Output dataset

In [10]:
final_dataset.to_json(
    "code_contests_plus_filtered.jsonl",
    orient="records",
    lines=True
)


Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

41770978